# B · 정혜원 — Linear SVM + ModernBERT 대조 학습 (Kaggle)

- 코드: `hyewon` 브랜치의 `experiments/B_svm.py`, `experiments/B_modernbert.py`
- train으로 학습, val로 비교. **test는 열지 않음.**
- 오른쪽 Settings: **Accelerator = GPU T4 x2**, **Internet = On** (휴대폰 인증 필요). P100은 최신 PyTorch와 호환 문제가 있을 수 있어 T4 권장.
- **Save Version → Save & Run All (Commit)** 으로 실행하면 브라우저를 닫아도 서버에서 끝까지 돈다. 결과는 버전의 Output 탭의 `B_results.zip`.

## 1. 저장소 받기·환경 기록

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging && git clone -q -b hyewon https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline
import platform, sys, numpy, pandas, sklearn
print(platform.platform(), '| Python', sys.version.split()[0], '| numpy', numpy.__version__,
      '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU'

## 2. Linear SVM (B-1 ~ B-4)
문체 32개 C=0.1/1/10 → 문체/문자 TF-IDF/결합 비교 → 스케일링 유무 비교. 확률은 train 내부 prompt_id GroupKFold(5) sigmoid 보정.

In [ ]:
!python -u -W ignore -m experiments.B_svm

## 3. SVM 결과 보기

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
m = pd.read_csv('results/course/B/svm/metrics.csv')
display(m[['run_id','svm_train_BA','svm_val_BA','svm_val_acc','svm_val_macro_f1','model_BA','company_BA','cal_model_LogLoss','converged','fit_seconds']].round(4))
print('C 고정 비교'); display(pd.read_csv('results/course/B/svm/feature_compare_fixedC.csv').round(4))
print('입력별 C 재튜닝'); display(pd.read_csv('results/course/B/svm/feature_compare_retunedC.csv').round(4))

## 4. ModernBERT-base (B-5 기본형 / B-6 대조 학습)
GPU 필요. 두 실행은 lam(대조 손실 가중치)만 다르고 나머지 조건은 동일. T4 기준 각각 수십 분 예상(미측정).

In [ ]:
import torch; assert torch.cuda.is_available(), 'Settings에서 Accelerator를 GPU로 바꾼 뒤 다시 실행'
print(torch.__version__, torch.cuda.get_device_name(0))
!pip -q install -U 'transformers>=4.48'

In [ ]:
!python -u -W ignore -m experiments.B_modernbert --lam 0

In [ ]:
!python -u -W ignore -m experiments.B_modernbert --lam 0.1

In [ ]:
import json, glob
rows = [json.load(open(p)) for p in sorted(glob.glob('results/course/B/modernbert/*/summary.json'))]
display(pd.DataFrame(rows)[['run_id','best_epoch','model_BA','accuracy','macro_f1','company_BA','cal_sibling_Acc','cal_model_LogLoss','val_silhouette_cos','fit_seconds','peak_gpu_mem_GB','gpu']].round(4))

## 5. 결과 저장
`/kaggle/working/B_results.zip` 이 버전 Output에 남는다. 받아서 Claude에게 주면 hyewon 브랜치에 커밋하고 main docs에 정리함.

In [ ]:
!zip -qr /kaggle/working/B_results.zip results/course/B preds/val_B_*
!ls -la /kaggle/working/B_results.zip